# 开盘区间交易模拟

检验上一轮（《开盘区间与高低点》）的思路能不能赚钱（决策 0016）：开盘 n 分钟的区间形成以后，

| 规则 | 做法 |
|---|---|
| 边缘反向 | 第一次碰到区间上沿做空、下沿做多（限价单挂在边缘），止损在边缘之外 `stop_adr` |
| 假突破收回 | 突破以后，第一根收盘回到区间里的 1 分钟 bar 收盘时反向入场，止损在突破以来的极值之外 `stop_adr` |
| 突破跟随（对照） | 第一次突破上沿做多、下沿做空，止损在区间中线（或对边），持有到收盘 |

目标：区间中线、区间对边，或不设目标持有到收盘。每只证券每天最多一笔，15:00 以后不开新仓，15:55 平仓。

**这是快速近似模拟，不是正式回测**：只用 1 分钟 bar；同一根 bar 里止损和目标都到了按先止损算；入场那根 bar 不算止盈；
止损单、市价单另加 0.01 ADR 滑点；成本用报表的分档（价格 > 100 美元单边 3 基点，20–100 美元 5 基点），买卖各扣一次。
数据与《日内波动报表》相同的清洗和复权。

**怎么读**：`avg_pct` 每笔平均收益（%，扣成本）；`gross_pct` 不扣成本（仍含滑点）；`win` 盈利笔数占比；
`pf` 盈亏比 = 总盈利 / 总亏损（> 1 才赚钱）；`avg_r` 平均每笔赚了几倍风险。

**用法**：改参数单元格，然后 *Run → Run All Cells*。第一次读 1 分钟 bar 约 2 分钟，之后读缓存（`reports/or_trades_cache/`）。
特征（开盘区间宽度、盘前振幅等）复用《开盘区间与高低点》的缓存；参数和那个 notebook 一样时不用重算。

In [1]:
# ===== 参数 =====
from trader.analytics import ReportParams
from trader.analytics.or_trades import Rule

tickers = ["NVDA", "AMD", "TSLA", "AAPL", "META", "AMZN", "GOOG", "MSFT", "SMH", "SNDK"]
ref = "QQQ"
start_date = "2024-01-02"
end_date = "2026-10-15"
focus = ["2026Q1", "2026Q2", "2026Q3", "2026Q4"]  # 重点看的季度（2026Q4 只到 10-15）
train_end = "2026Q1"  # 分档边界只用这个季度以前的交易定（2024–2025），再用到 2026

# 主要比较的规则；Rule(种类, n=开盘区间分钟, target=目标, stop_adr=止损距离,
#                    stop_at=突破跟随的止损位置)
rules = [
    Rule("fade", n=60, target="mid"),
    Rule("fade", n=60, target="opposite"),
    Rule("reclaim", n=60, target="mid"),
    Rule("reclaim", n=60, target="opposite"),
    Rule("breakout", n=60, stop_at="mid"),
]

# 和《开盘区间与高低点》相同的报表参数（复用它的缓存）
params = ReportParams(
    benchmark_tickers=(ref, "SPY"),
    opening_windows_min=(30, 60, 90),
    open_skip_compare=(0, 5, 15, 30, 60, 90),
    include_extended_hours=True,
)
use_cache = True

In [2]:
import hashlib
import pickle

import pandas as pd
from IPython.display import Markdown, display

from trader.analytics import build_report, opening
from trader.analytics import or_trades as T
from trader.config import project_dir

pd.set_option("display.max_rows", 200)
pct = "{:.0%}"
FMT = {
    "trades": "{:.0f}",
    "win": pct,
    "avg_pct": "{:+.3f}",
    "gross_pct": "{:+.3f}",
    "avg_adr": "{:+.3f}",
    "avg_r": "{:+.2f}",
    "pf": "{:.2f}",
    "sum_pct": "{:+.1f}",
    "target": pct,
    "stop": pct,
    "close": pct,
    "hold": "{:.0f}",
}
COLS = ["trades", "win", "avg_pct", "gross_pct", "avg_r", "pf", "target", "stop", "close", "hold"]


def fmt(t, cols=COLS):
    t = t[[c for c in cols if c in t.columns]]
    return t.style.format({k: v for k, v in FMT.items() if k in t.columns}).background_gradient(
        subset=[c for c in ("avg_pct",) if c in t.columns], cmap="RdYlGn", vmin=-0.2, vmax=0.2
    )


root = project_dir() / "reports"
# 1 分钟 bar
mkey = hashlib.sha1(repr((sorted(tickers), start_date, end_date)).encode()).hexdigest()[:12]
mcache = root / "or_trades_cache" / f"minutes_{start_date}_{end_date}_{mkey}.pkl"
if use_cache and mcache.exists():
    days = pickle.loads(mcache.read_bytes())
    print(f"读取缓存 {mcache.name}")
else:
    days = T.load_minutes(tickers, start_date, end_date)
    mcache.parent.mkdir(parents=True, exist_ok=True)
    mcache.write_bytes(pickle.dumps(days))
    print(f"读取完成，已存缓存 {mcache.name}")
# 第 n 分钟时已知的特征（与《开盘区间与高低点》同一份报表缓存）
key = hashlib.sha1(repr((sorted(tickers), start_date, end_date, params)).encode()).hexdigest()[:12]
rcache = root / "opening_cache" / f"{start_date}_{end_date}_{key}.pkl"
if use_cache and rcache.exists():
    rep = pickle.loads(rcache.read_bytes())
else:
    rep = build_report(tickers, start_date, end_date, params)
    rep.history = None
    rcache.parent.mkdir(parents=True, exist_ok=True)
    rcache.write_bytes(pickle.dumps(rep))
print(f"{len(days)} 个证券日（已去掉提前收盘日和 ADR 不足的日子）")

trades = {r.label: T.add_period(T.simulate(days, r)) for r in rules}

读取缓存 minutes_2024-01-02_2026-10-15_5ecc96848ebd.pkl
6447 个证券日（已去掉提前收盘日和 ADR 不足的日子）


## 1. 各规则按季度

每条规则一张表，每季度一行，最后一行"全部"。绿色 = 扣成本后赚钱，红色 = 亏钱。先看 2026 年各季度，再和 2024–2025 对比：
只在个别季度为正、换个季度就转负的，不能当成规律。

In [3]:
for label, t in trades.items():
    display(Markdown(f"**{label}**"))
    display(fmt(T.summarize(t)))

**边缘反向 n=60 目标区间中线 止损0.15ADR**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
quarter,,,,,,,,,,
2024Q1,298,30%,-0.107,-0.047,-0.32,0.63,22%,67%,10%,72
2024Q2,476,27%,-0.144,-0.084,-0.46,0.54,18%,70%,12%,72
2024Q3,483,30%,-0.154,-0.094,-0.34,0.57,27%,68%,5%,54
2024Q4,461,31%,-0.053,+0.007,-0.21,0.81,25%,66%,9%,72
2025Q1,473,28%,-0.169,-0.108,-0.36,0.55,22%,68%,9%,67
2025Q2,548,34%,-0.131,-0.066,-0.28,0.66,28%,64%,8%,62
2025Q3,519,31%,-0.081,-0.018,-0.27,0.71,24%,65%,11%,70
2025Q4,513,31%,-0.121,-0.061,-0.30,0.67,25%,66%,9%,63
2026Q1,513,34%,-0.044,+0.016,-0.20,0.87,27%,63%,10%,75


**边缘反向 n=60 目标区间对边 止损0.15ADR**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
quarter,,,,,,,,,,
2024Q1,298,25%,-0.086,-0.026,-0.30,0.72,7%,71%,22%,100
2024Q2,476,24%,-0.148,-0.088,-0.45,0.54,5%,73%,23%,98
2024Q3,483,26%,-0.144,-0.084,-0.29,0.62,10%,72%,18%,91
2024Q4,461,26%,-0.082,-0.022,-0.30,0.71,5%,69%,26%,109
2025Q1,473,24%,-0.176,-0.115,-0.39,0.55,7%,71%,22%,98
2025Q2,548,28%,-0.133,-0.067,-0.28,0.68,12%,69%,19%,93
2025Q3,519,27%,-0.094,-0.031,-0.30,0.68,5%,68%,26%,104
2025Q4,513,25%,-0.130,-0.070,-0.34,0.66,8%,69%,23%,98
2026Q1,513,27%,-0.072,-0.012,-0.27,0.80,8%,69%,23%,109


**假突破收回 n=60 目标区间中线 止损0.15ADR**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
quarter,,,,,,,,,,
2024Q1,278,44%,-0.027,+0.033,-0.11,0.91,30%,51%,19%,108
2024Q2,442,38%,-0.106,-0.046,-0.28,0.68,26%,54%,21%,104
2024Q3,455,45%,-0.104,-0.044,-0.10,0.74,36%,48%,16%,93
2024Q4,430,42%,-0.021,+0.039,-0.10,0.93,33%,52%,14%,99
2025Q1,450,42%,-0.081,-0.020,-0.16,0.79,32%,51%,17%,95
2025Q2,509,47%,-0.056,+0.010,-0.07,0.86,38%,45%,17%,90
2025Q3,490,42%,-0.038,+0.025,-0.12,0.87,33%,49%,19%,98
2025Q4,480,40%,-0.103,-0.043,-0.16,0.74,32%,51%,17%,90
2026Q1,485,44%,-0.023,+0.037,-0.09,0.94,32%,50%,18%,102


**假突破收回 n=60 目标区间对边 止损0.15ADR**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
quarter,,,,,,,,,,
2024Q1,278,38%,+0.003,+0.063,-0.08,1.01,9%,54%,37%,146
2024Q2,442,32%,-0.154,-0.094,-0.33,0.58,6%,58%,36%,140
2024Q3,455,40%,-0.085,-0.025,-0.04,0.80,13%,51%,36%,141
2024Q4,430,36%,-0.038,+0.022,-0.16,0.88,7%,55%,38%,146
2025Q1,450,36%,-0.096,-0.035,-0.19,0.77,9%,55%,36%,138
2025Q2,509,39%,-0.052,+0.013,-0.06,0.88,17%,50%,33%,132
2025Q3,490,37%,-0.035,+0.028,-0.14,0.89,8%,52%,40%,144
2025Q4,480,34%,-0.120,-0.060,-0.18,0.72,10%,55%,35%,134
2026Q1,486,36%,-0.052,+0.008,-0.14,0.87,10%,55%,34%,143


**突破跟随 n=60 止损中线**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
quarter,,,,,,,,,,
2024Q1,298,36%,-0.206,-0.146,-0.25,0.54,0%,37%,63%,212
2024Q2,476,43%,-0.015,+0.045,+0.02,0.96,0%,31%,69%,222
2024Q3,483,39%,-0.032,+0.028,-0.11,0.93,0%,40%,60%,209
2024Q4,461,39%,-0.156,-0.096,-0.14,0.64,0%,37%,63%,218
2025Q1,473,41%,-0.075,-0.013,-0.10,0.85,0%,37%,63%,201
2025Q2,548,33%,-0.062,+0.004,-0.18,0.89,0%,44%,56%,189
2025Q3,519,37%,-0.161,-0.097,-0.19,0.62,0%,38%,62%,206
2025Q4,513,42%,-0.002,+0.058,-0.01,1.00,0%,37%,63%,202
2026Q1,513,40%,-0.122,-0.062,-0.14,0.76,0%,35%,65%,215


## 2. 2026 年各季度并排

只看 `focus` 里的季度：每条规则的每笔平均收益（扣成本 / 不扣成本）和盈亏比。

In [4]:
side = {}
for label, t in trades.items():
    s = T.summarize(t[t["quarter"].isin(focus)])
    side[label] = s[["trades", "avg_pct", "gross_pct", "pf"]]
side = pd.concat(side, axis=1)
display(
    side.style.format("{:+.3f}")
    .format("{:.0f}", subset=[c for c in side.columns if c[1] == "trades"])
    .format("{:.2f}", subset=[c for c in side.columns if c[1] == "pf"])
)

## 3. 只在波动大（或小）的日子做，有没有帮助？

按第 n 分钟已经知道的特征分三档：`or_width` 开盘区间宽度、`pm_range` 盘前振幅、`vol5` 最近 5 天平均振幅（都以 ADR 为单位）。
**分档边界只用 2024–2025 的交易定**，再原样用到 2026 年，相当于"用过去定规则、在之后检验"。
上半部分是 2024–2025（定边界的那段），下半部分是 2026。两段方向一致才有参考价值。

In [5]:
filt_features = ["or_width", "pm_range", "vol5"]
for r in rules:
    feats = opening.features(rep.daily, r.n, ref=ref)
    t = T.with_features(trades[r.label], feats, filt_features)
    train = t["quarter"] < train_end
    t["段"] = ["2024–2025" if x else "2026" for x in train]
    display(Markdown(f"**{r.label}**"))
    for f in filt_features:
        b = T.fixed_bins(t, f, train)
        t["档"] = b.astype(str)
        s = T.summarize(t[b.notna().to_numpy()], ["段", "档"])
        s.index.names = ["段", f]
        display(fmt(s, ["trades", "win", "avg_pct", "gross_pct", "pf"]))

**边缘反向 n=60 目标区间中线 止损0.15ADR**

**边缘反向 n=60 目标区间对边 止损0.15ADR**

**假突破收回 n=60 目标区间中线 止损0.15ADR**

**假突破收回 n=60 目标区间对边 止损0.15ADR**

**突破跟随 n=60 止损中线**

## 4. 参数网格：区间长度 × 止损 × 目标

每个组合一行：2024–2025 与 2026 两段各自的每笔平均收益（扣成本 / 不扣成本）和 2026 的盈亏比。
组合很多时，总会有几个碰巧为正，要看它们是不是两段都为正、而且相邻的参数也差不多。

In [6]:
grid = {}
for kind in ["fade", "reclaim"]:
    for n_ in [30, 60, 90]:
        for st in [0.1, 0.15, 0.25, 0.4]:
            for tg in ["mid", "opposite"]:
                t = T.add_period(T.simulate(days, Rule(kind, n=n_, stop_adr=st, target=tg)))
                t["段"] = ["2024–2025" if q < train_end else "2026" for q in t["quarter"]]
                s = T.summarize(t, "段")
                grid[(kind, n_, st, tg)] = {
                    "2024–25 avg_pct": s.loc["2024–2025", "avg_pct"],
                    "2024–25 gross_pct": s.loc["2024–2025", "gross_pct"],
                    "2026 avg_pct": s.loc["2026", "avg_pct"],
                    "2026 gross_pct": s.loc["2026", "gross_pct"],
                    "2026 pf": s.loc["2026", "pf"],
                    "trades": s.loc["全部", "trades"],
                }
grid = pd.DataFrame(grid).T
grid.index.names = ["规则", "n", "止损ADR", "目标"]
display(
    grid.style.format("{:+.3f}")
    .format("{:.2f}", subset=["2026 pf"])
    .format("{:.0f}", subset=["trades"])
    .background_gradient(
        subset=["2024–25 avg_pct", "2026 avg_pct"], cmap="RdYlGn", vmin=-0.15, vmax=0.15
    )
)

## 5. 按证券

第一条"假突破收回"规则（本页规则列表里的第 3 条），全部季度合计，每只证券一行。

In [7]:
lab = rules[2].label
display(Markdown(f"**{lab}**"))
display(fmt(T.summarize(trades[lab], "symbol")))

**假突破收回 n=60 目标区间中线 止损0.15ADR**

,trades,win,avg_pct,gross_pct,avg_r,pf,target,stop,close,hold
symbol,,,,,,,,,,
AAPL,530,39%,-0.081,-0.021,-0.24,0.70,31%,55%,15%,99
AMD,535,46%,-0.016,+0.046,-0.02,0.96,34%,46%,21%,101
AMZN,533,45%,-0.028,+0.032,-0.07,0.90,36%,48%,16%,96
GOOG,508,44%,-0.026,+0.034,-0.08,0.90,33%,49%,19%,102
META,526,45%,-0.027,+0.033,-0.09,0.91,35%,48%,18%,98
MSFT,529,40%,-0.070,-0.010,-0.19,0.71,33%,53%,15%,93
NVDA,522,39%,-0.143,-0.083,-0.22,0.67,28%,52%,20%,95
SMH,547,40%,-0.086,-0.026,-0.18,0.72,33%,51%,16%,89
SNDK,291,41%,-0.100,-0.027,-0.09,0.88,29%,52%,20%,106


## 6. 导出

`reports/or_trades_<起>_<止>/`：trades_<规则序号>.csv（每笔交易）、by_quarter.csv（各规则按季度）、grid.csv（参数网格）。

In [8]:
outdir = root / f"or_trades_{start_date}_{end_date}"
outdir.mkdir(parents=True, exist_ok=True)
allq = []
for i, (label, t) in enumerate(trades.items(), 1):
    t.to_csv(outdir / f"trades_{i}.csv", index=False, encoding="utf-8-sig")
    allq.append(T.summarize(t).assign(rule=label))
pd.concat(allq).to_csv(outdir / "by_quarter.csv", encoding="utf-8-sig")
grid.to_csv(outdir / "grid.csv", encoding="utf-8-sig")
print(outdir)

C:\My_Works\DayTrader\DayTrader\reports\or_trades_2024-01-02_2026-10-15
